# Step 3: cell type and pseudotime transferred from mouse to human, and Samap_Final.h5ad

Both transfers use the human x mouse block `C` of the joint SAMap graph (`obsp['connectivities']` of step 2; rows are
human nuclei, columns mouse nuclei, entries the SAMap edge weights). Each human nucleus takes the values of the mouse
nuclei it is joined to, weighted by the edge weights (the Methods' kernel-weighted average):

- **Cell type**: the mouse `cell_type` carrying most of the nucleus's edge weight to mouse (`label_stain_full.py` of
  the run of record); `unmapped` when the nucleus has no mouse edge. A tie would go to `decidual_cell`, the first
  label in sorted order (there are none in the run of record).
- **Pseudotime**: the edge-weighted mean of the mouse pseudotime of record, rank-normalized (rank / N over the mouse
  nuclei with a value), over the mouse neighbors that have one (`samap_transfer_pt.py`, variant `pt_ranknorm`).
  decAp nuclei have no pseudotime, so a human nucleus whose mouse neighbors are all decAp gets none. The final mouse
  object stores its pseudotime as rank / N already, so the rank normalization changes nothing.

The first cell after the parameters is both transfers. The rest writes `Samap_Final.h5ad`, the deposited object: the
joint object with its obs replaced by six columns (`species`; `sample`, `timepoint` and `cell_type` from the final
objects; `transferred_mouse_pseudotime`; `transferred_mouse_fibdec`, missing where unmapped and on mouse nuclei),
every other dataset copied unchanged (`slim_samap_obs.py` of the run of record); then `transfers.csv` (one row per
human nucleus) and `summary.json` (the counts). Written to `OUT/3_transfer.partial/`, renamed when complete.

In [ ]:
# Parameters; papermill -p NAME VALUE overrides them in a cell below this one.
OUT = ""  # the run folder; this step writes OUT/3_transfer/
JOINT = ""  # the joint SAMap object; "" means OUT/2_samap/samap_joint.h5ad (step 2)
HUMAN_H5AD = "/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MOUSE_H5AD = "/n/netscratch/eddy_lab/Everyone/nhilgert/final_objects/Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"

## The transfers

In [ ]:
import os
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
from anndata.io import read_elem, write_elem

assert OUT, "set OUT, the run folder"
joint_path = Path(JOINT or Path(OUT) / "2_samap" / "samap_joint.h5ad")
out_dir = Path(OUT) / "3_transfer"
partial_dir = Path(OUT) / "3_transfer.partial"
assert not out_dir.exists(), f"{out_dir} exists; step 3 is done"
partial_dir.mkdir(parents=True, exist_ok=True)

# The joint graph, and its human x mouse block C.
with h5py.File(joint_path, "r") as f:
    joint_obs = read_elem(f["obs"])
    graph = sp.csr_matrix(read_elem(f["obsp/connectivities"]))
names = joint_obs.index.astype(str)
species = joint_obs["species"].astype(str).to_numpy()
is_human, is_mouse = species == "hs", species == "mm"
C = graph[is_human][:, is_mouse].tocsr()
print(f"joint object {joint_path}: {len(names)} nuclei ({is_human.sum()} human, {is_mouse.sum()} mouse); "
      f"human x mouse block {C.shape}, {C.nnz} edges")

# The objects of 6-10 Nov 2026 label the compartments eFib and dec; the run of record, the later steps and the
# figures use these names.
LABEL_MAP = {"eFib": "endometrial_fibroblast", "dec": "decidual_cell"}

# Labels and pseudotime of record from the final objects, in the graph's row (human) and column (mouse) order.
final = {}
for code, path in {"hs": HUMAN_H5AD, "mm": MOUSE_H5AD}.items():
    with h5py.File(path, "r") as f:
        final[code] = read_elem(f["obs"])[["sample", "timepoint", "cell_type", "pseudotime"]]
    labels = final[code]["cell_type"].astype(str).map(lambda v: LABEL_MAP.get(v, v))
    final[code]["cell_type"] = labels.astype("category")
    in_joint = names[species == code]
    assert len(final[code]) == len(in_joint) and in_joint.isin(final[code].index).all(), \
        f"{path} and the joint object hold different {code} nuclei"
human = final["hs"].loc[names[is_human]]
mouse = final["mm"].loc[names[is_mouse]]

# Cell type: each human nucleus's edge weight to mouse, split by mouse cell type (C @ onehot); the label with the
# most weight wins. categories are sorted, so a tie goes to the first, as in the run of record.
labels = mouse["cell_type"].astype(str).to_numpy()
categories = np.unique(labels)
codes = pd.Categorical(labels, categories=categories).codes
onehot = sp.csr_matrix((np.ones(len(labels)), (np.arange(len(labels)), codes)), shape=(len(labels), categories.size))
weight_by_label = np.asarray((C @ onehot).todense())
cross_species_weight = weight_by_label.sum(axis=1)
has_edges = cross_species_weight > 1e-9
fibdec = np.where(has_edges, categories[weight_by_label.argmax(axis=1)], "unmapped")
vote_share = weight_by_label.max(axis=1) / np.maximum(cross_species_weight, 1e-9)
tie = has_edges & ((weight_by_label == weight_by_label.max(axis=1, keepdims=True)).sum(axis=1) > 1)

# Pseudotime: rank / N over the mouse nuclei with a value (NaN stays NaN), then the edge-weighted mean over each
# human nucleus's mouse neighbors with a value (Nadaraya-Watson: C @ values / C @ has_value); NaN where none.
mouse_pt = mouse["pseudotime"].to_numpy(dtype=float)
has_value = np.isfinite(mouse_pt)
mouse_pt_rank = np.full(len(mouse_pt), np.nan)
mouse_pt_rank[has_value] = pd.Series(mouse_pt[has_value]).rank(method="average").to_numpy() / has_value.sum()
print(f"mouse pseudotime: {has_value.sum()} of {len(mouse_pt)} nuclei; rank normalization changes it by "
      f"at most {np.nanmax(np.abs(mouse_pt_rank - mouse_pt)):.1e}")
numerator = C @ np.where(has_value, mouse_pt_rank, 0.0)
pt_weight = C @ has_value.astype(float)
transferred_pt = np.full(C.shape[0], np.nan)
ok = pt_weight > 1e-10
transferred_pt[ok] = numerator[ok] / pt_weight[ok]

transfers = pd.DataFrame({
    "native_cell_type": human["cell_type"].astype(str).to_numpy(),
    "transferred_mouse_fibdec": fibdec,
    "fibdec_vote_share": vote_share,
    "cross_species_weight": cross_species_weight,
    "transferred_mouse_pseudotime": transferred_pt,
    "pseudotime_weight": pt_weight,
}, index=pd.Index(names[is_human], name="cell"))
transfers.to_csv(partial_dir / "transfers.csv")
print(transfers["transferred_mouse_fibdec"].value_counts().to_dict(),
      f"{int(np.isfinite(transferred_pt).sum())} human nuclei with a transferred pseudotime")

## Samap_Final.h5ad

A copy of the joint object with the obs of `slim_samap_obs.py`: categorical sample, timepoint and cell_type over both
species, each species' values and category order from its final object (human categories first); the transferred
cell type as a categorical over the same cell types. Only obs and the provenance note change; every other dataset
stays as step 2 wrote it.

In [ ]:
import datetime
import hashlib
import json
import shutil


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


def combined(column):
    """One categorical over both species, in joint order; categories from the human object, then the mouse one."""
    categories = list(dict.fromkeys([*final["hs"][column].cat.categories, *final["mm"][column].cat.categories]))
    values = pd.concat([final["hs"][column].astype(str), final["mm"][column].astype(str)]).reindex(names)
    return pd.Categorical(values.to_numpy(), categories=categories)


cell_types = combined("cell_type").categories
pt_column = np.full(len(names), np.nan)
pt_column[is_human] = transferred_pt
# 'unmapped' and the mouse nuclei, which are not among the categories, become missing values.
fibdec_column = np.full(len(names), None, dtype=object)
fibdec_column[is_human] = fibdec

new_obs = pd.DataFrame({
    "species": pd.Categorical(species, categories=["hs", "mm"]),
    "sample": combined("sample"),
    "timepoint": combined("timepoint"),
    "cell_type": combined("cell_type"),
    "transferred_mouse_pseudotime": pt_column,
    "transferred_mouse_fibdec": pd.Categorical(fibdec_column, categories=list(cell_types)),
}, index=pd.Index(names.to_numpy(), dtype=object))

target = partial_dir / "Samap_Final.h5ad"
shutil.copyfile(joint_path, target)
with h5py.File(target, "r+") as f:
    provenance = json.loads(read_elem(f["uns/provenance"])) if "uns/provenance" in f else {}
    provenance["samap_final"] = {
        "written": datetime.datetime.now().isoformat(timespec="seconds"),
        "step": "samap step 3 (notebooks/3_transfer.ipynb)",
        "joint": str(joint_path),
        "obs": ("species; sample, timepoint, cell_type from the final objects; transferred_mouse_pseudotime = "
                "edge-weighted mean of the rank-normalized mouse pseudotime over each human nucleus's mouse "
                "neighbors with one; transferred_mouse_fibdec = mouse cell_type with most edge weight"),
        "final_objects": {code: {"path": str(path), "sha256": sha256_of(path)}
                          for code, path in {"hs": HUMAN_H5AD, "mm": MOUSE_H5AD}.items()},
    }
    del f["obs"]
    write_elem(f, "obs", new_obs)
    if "uns/provenance" in f:
        del f["uns/provenance"]
    write_elem(f["uns"], "provenance", json.dumps(provenance))


## Bookkeeping: the counts, a read-back check; the step is complete

In [ ]:
labelled = transfers["transferred_mouse_fibdec"] != "unmapped"
agrees = labelled & (transfers["transferred_mouse_fibdec"] == transfers["native_cell_type"])
no_mouse_edge = transfers["cross_species_weight"] <= 1e-9
no_pt = transfers["transferred_mouse_pseudotime"].isna()
summary = {
    "human_nuclei": int(len(transfers)),
    "cell_type": {
        "labelled": int(labelled.sum()),
        "unmapped": int((~labelled).sum()),
        "ties": int(tie.sum()),
        "median_vote_share": float(transfers.loc[labelled, "fibdec_vote_share"].median()),
        "counts": {k: int(v) for k, v in transfers.loc[labelled, "transferred_mouse_fibdec"].value_counts().items()},
        "agree_with_native": int(agrees.sum()),
        "agree_share_of_labelled": round(float(agrees.sum() / labelled.sum()), 4),
        "agree_share_of_all": round(float(agrees.mean()), 4),
        "agree_by_native": {
            native: {"nuclei": int(rows.sum()), "agree": int((agrees & rows).sum()),
                     "share": round(float((agrees & rows).sum() / rows.sum()), 4)}
            for native, rows in ((n, transfers["native_cell_type"] == n) for n in sorted(transfers["native_cell_type"].unique()))
        },
    },
    "pseudotime": {
        "with_value": int((~no_pt).sum()),
        "without_value": int(no_pt.sum()),
        "without_value_no_mouse_edge": int((no_pt & no_mouse_edge).sum()),
        "without_value_only_mouse_neighbors_without_pseudotime": int((no_pt & ~no_mouse_edge).sum()),
        "median_by_timepoint": {tp: round(float(transfers.loc[(human["timepoint"] == tp).to_numpy(),
                                                              "transferred_mouse_pseudotime"].median()), 4)
                                for tp in sorted(human["timepoint"].astype(str).unique())},
    },
}
print(json.dumps(summary, indent=1))

# The obs read back must equal what was written, column by column.
with h5py.File(target, "r") as f:
    back = read_elem(f["obs"])
assert list(back.columns) == list(new_obs.columns) and back.index.astype(str).equals(names)
for column in new_obs.columns:
    if new_obs[column].dtype.kind == "f":
        assert np.array_equal(back[column].to_numpy(), new_obs[column].to_numpy(), equal_nan=True), column
    else:
        assert back[column].astype(str).equals(new_obs[column].astype(str)), column

summary["samap_final"] = {"bytes": target.stat().st_size, "obs_columns": list(new_obs.columns),
                          "joint": str(joint_path), "joint_sha256": sha256_of(joint_path)}
(partial_dir / "summary.json").write_text(json.dumps(summary, indent=1) + "\n")
os.replace(partial_dir, out_dir)
print(f"wrote {out_dir}: {sorted(p.name for p in out_dir.iterdir())}")
